In [ ]:
from traceback import print_list
from tracemalloc import start
from numpy import save
import pandas as pd
import polars as pl
from pandas import ExcelWriter
#import re
import xlsxwriter
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import load_workbook,Workbook,worksheet,utils
from openpyxl.utils.dataframe import dataframe_to_rows
from procyclingstats import Race, Rider, Stage,stage_scraper

enlace_o_valor="race/gran-premio-miguel-indurain/2026"

     # esto para carreras de un solo día  
race= Race(f"{enlace_o_valor}/overview") 
file=f"analisis_carrera{race.name()}.xlsx"
#libros=xlsxwriter.Workbook(file)
hojas=libros.add_worksheet() # type: ignore
hoja=libro.Workbook() # type: ignore
#writer=pd.ExcelWriter(file, engine='xlsxwriter') # type: ignore
#libro=writer.book
#hoja=writer.sheets
libros=Workbook()
hojas=libros.active

df_res = pd.DataFrame()  
df_fin = pd.DataFrame()
list_editions = race.prev_editions_select() 
c=0

hojas.write(0,0,race.name()) # type: ignore
#hoja['A1'].format = Font(color='FF0000', bold=True,size=25) # type: ignore
for edition in list_editions[1:6]:
   
    print("Procesando edición", edition['text'])
    hojas.write(c+1,0,f"Procesando edición {edition['text']}")
    past_edit=Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/result")
    print("  Obteniendo resultados..."+f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/result")
    #print(past_edit.parse().keys())
    print(f'como:{past_edit.won_how()},participacion:{past_edit.race_startlist_quality_score()} ,desnivel: {past_edit.vertical_meters()}m ,distancia {past_edit.distance()}km, avg:{past_edit.avg_speed_winner()}km/h')
    hojas.write(c+2,0,f'como:{past_edit.won_how()}')
    hojas.write(c+2,1, f'participacion:{past_edit.race_startlist_quality_score()}' )
    hojas.write(c+2,2,f'desnivel: {past_edit.vertical_meters()}m')
    hojas.write(c+2,3,f'distancia {past_edit.distance()}km' )
    hojas.write(c+2,4,f'avg:{past_edit.avg_speed_winner()}km/h')
    df_res = pd.DataFrame(past_edit.parse()['results'])
    
    if(df_res.empty):
        print("  No hay resultados para esta edición.")
        hojas.write(c+2,0,"  No hay resultados para esta edición.")
        continue
    
    rider=Rider(str(df_res['rider_url'].head(1).values[0])) # type: ignore

    
    points_per_speciality = rider.parse()['points_per_speciality']
    if isinstance(points_per_speciality, dict):
        sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
        df_rider_specialidad = pd.DataFrame([sorted_by_values])
    else:
        df_rider_specialidad = pd.DataFrame(points_per_speciality)
   
    print(df_rider_specialidad.keys()[0]+":"+str(df_rider_specialidad.iloc[0,0])+","+
          df_rider_specialidad.keys()[1]+":"+str(df_rider_specialidad.iloc[0,1]))
    for r_idx, row in enumerate(dataframe_to_rows(df_res, header=True, index=False)):
        hoja.append(row) 
    #df_res.to_excel(file,startrow=c+4,sheet_name='Sheet', header=True, index=False,engine='xlsxwriter')    
    '''
    with pd.ExcelWriter(file, engine='xlsxwriter') as writer:
        
        df_res[['rank', 'rider_name','time','team_name']].head(5).to_excel(writer,startrow=c+4,sheet_name='Sheet', header=True, index=False)
        df=df_res.groupby('team_name')['uci_points'].sum().reset_index().sort_values(by='uci_points', ascending=False)
        df.to_excel(file,startrow=c+4,startcol=8,sheet_name='Sheet', header=True, index=False)
    
        df_burgos = df_res[df_res['team_name'].str.contains('Burgos')]
    
        df_burgos[['rank', 'rider_name','time','uci_points','breakaway_kms']].head(4).to_excel(file,startrow=c+10,sheet_name='Sheet', header=True, index=False)
     '''
    c=c+15
   
libros.save(FileExistsError)  


Procesando edición 2025
  Obteniendo resultados...race/gran-premio-miguel-indurain/2025/result
como:1.6 km solo,participacion:(393, 393) ,desnivel: 3351m ,distancia 203.9km, avg:40.794km/h
hills:736,one_day_races:670
Procesando edición 2024
  Obteniendo resultados...race/gran-premio-miguel-indurain/2024/result
como:Sprint à deux,participacion:(381, 381) ,desnivel: 3178m ,distancia 198.1km, avg:40.183km/h
gc:2888,time_trial:2391
Procesando edición 2023
  Obteniendo resultados...race/gran-premio-miguel-indurain/2023/result
como:1.9 km solo,participacion:(297, 297) ,desnivel: 3445m ,distancia 203.2km, avg:39.45km/h
gc:5471,climber:4714
Procesando edición 2022
  Obteniendo resultados...race/gran-premio-miguel-indurain/2022/result
como:Sprint of small group,participacion:(371, 371) ,desnivel: 2878m ,distancia 190.0km, avg:38.279km/h
climber:3987,gc:3121
Procesando edición 2021
  Obteniendo resultados...race/gran-premio-miguel-indurain/2021/result
como:2.1 km solo,participacion:(365, 365) ,d

In [ ]:
#carreras por etapas
enlace_o_valor="race/itzulia-basque-country/2026"
import gc
 

print("Intentando obtener resultados para:", enlace_o_valor)
race= Race(f"{enlace_o_valor}/overview")     
df_res = pd.DataFrame()  
df_fin = pd.DataFrame()
list_editions = race.prev_editions_select() 
etapas=race.stages()
print(list_editions)

for edition in list_editions[1:10]:
   
    race= Race(f"{edition['value']}/overview")  
    print("========================================")
    print("Procesando edición", edition['text'])
    print("========================================")
    etapas=race.stages()
    for etapa in etapas:
        print(f"{etapa['stage_url']}/result")
        past_edit=Stage(f"{etapa['stage_url']}/result")
        #print(past_edit.parse().keys())
        print(past_edit.departure()+" - "+past_edit.arrival())
        print(f'como:{past_edit.won_how()} ,participacion:{past_edit.race_startlist_quality_score()},desnivel: {past_edit.vertical_meters()}m ,distancia {past_edit.distance()}km, avg:{past_edit.avg_speed_winner()}km/h')
        df_res = pd.DataFrame(past_edit.parse()['results'])
    
        #print(df_res['rider_url'].head(1))
        rider=Rider(str(df_res['rider_url'].head(1).values[0])) # type: ignore
    
        points_per_speciality = rider.parse()['points_per_speciality']
        if isinstance(points_per_speciality, dict):
            sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
            df_rider_specialidad = pd.DataFrame([sorted_by_values])
        else:
            df_rider_specialidad = pd.DataFrame(points_per_speciality)
   
        print(df_rider_specialidad.keys()[0]+":"+str(df_rider_specialidad.iloc[0,0])+","+
        df_rider_specialidad.keys()[1]+":"+str(df_rider_specialidad.iloc[0,1]))
          
        print(df_res[['rank', 'rider_name','time','team_name']].head(5))
        df=df_res.groupby('team_name')['uci_points'].sum().reset_index().sort_values(by='uci_points', ascending=False)
        #print(df)
        df_burgos = df_res[df_res['team_name'].str.contains('Burgos')]
     
        print(df_burgos[['rank', 'rider_name', 'time', 'breakaway_kms','uci_points']].head(4))   
        gc.collect() 

       

Intentando obtener resultados para: race/itzulia-basque-country/2026


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="cmpwelcomebtnyes"]"}
  (Session info: chrome=143.0.7499.41); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff68c448245
	0x7ff68c4482a0
	0x7ff68c22165d
	0x7ff68c279a33
	0x7ff68c279d3c
	0x7ff68c2cdf67
	0x7ff68c2cac97
	0x7ff68c26ac29
	0x7ff68c26ba93
	0x7ff68c75ffe0
	0x7ff68c75a920
	0x7ff68c779086
	0x7ff68c465744
	0x7ff68c46e6ec
	0x7ff68c451964
	0x7ff68c451b15
	0x7ff68c437842
	0x7ffc69375c07
	0x7ffc6a679cc0
